## Setup
This notebook uses the local `.venv` virtual environment.  
**Select kernel:** `Python (Spradley)` in the top-right kernel picker before running.

In [5]:
import pandas as pd

In [6]:
df = pd.read_csv('pilot_transcripts.csv', parse_dates=['timestamp'])
df.head()

,session_id,turn_number,speaker,message,sentiment,sentiment_score,timestamp,initial_mood,final_mood
0,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,1,User,Great team and exciting new things coming up!,positive,90.0,2026-03-23 15:38:35.718+00,NaN,NaN
1,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,2,Bot,"When you say it's a 'great team,' what does th...",NaN,NaN,2026-03-23 15:38:35.718+00,NaN,NaN
2,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,3,User,supportive and super friendly atmosphere,positive,90.0,2026-03-23 15:39:02.48+00,NaN,NaN
3,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,4,Bot,Can you think of a specific time recently when...,NaN,NaN,2026-03-23 15:39:02.48+00,NaN,NaN
4,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,5,User,A colleague tested something for me while I wa...,positive,95.0,2026-03-23 15:40:04.987+00,NaN,NaN


In [12]:
ts = pd.to_datetime(df['timestamp'], utc=True, format='ISO8601')
print(f"Shape:          {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Sessions:       {df['session_id'].nunique()} pilot conversations")
print(f"Date range:     {ts.min().date()} → {ts.max().date()}")
print(f"Turns (each):   {df['speaker'].value_counts()['User']} user  |  {df['speaker'].value_counts()['Bot']} bot")
print()
print("User sentiment breakdown:")
print(df.dropna(subset=['sentiment'])['sentiment'].value_counts().to_string())
print()
print(f"Sentiment score: {df['sentiment_score'].min():.0f} – {df['sentiment_score'].max():.0f}  (mean {df['sentiment_score'].mean():.1f})")
print()
print("Columns with all nulls:", df.columns[df.isnull().all()].tolist())

Shape:          98 rows × 9 columns
Sessions:       5 pilot conversations
Date range:     2026-03-23 → 2026-03-25
Turns (each):   49 user  |  49 bot

User sentiment breakdown:
sentiment
positive    28
neutral     16
negative     5

Sentiment score: 20 – 95  (mean 68.0)

Columns with all nulls: ['initial_mood', 'final_mood']


In [ ]:
# ── C0: Config ───────────────────────────────────────────────────────────────
# Edit the values below before running. These override pipeline_core.py defaults.
# If you skip this cell, pipeline_core.py defaults apply automatically.

import pipeline_core

# ── Tunable per run — edit here ──────────────────────────────────────────────
pipeline_core.CONFIG["LLM_MODEL"]       = "claude-haiku-4-5-20251001"
pipeline_core.CONFIG["LLM_TEMPERATURE"] = 0.2
pipeline_core.CONFIG["L1_CODES_RANGE"]  = (1, 10)   # open codes per Q&A pair
pipeline_core.CONFIG["L2_CODES_RANGE"]  = (20, 30)  # consolidated codes per interview
pipeline_core.CONFIG["L3_CODES_RANGE"]  = (40, 80)  # consolidated codes across all employees
pipeline_core.CONFIG["CLUSTERS_RANGE"]  = (7, 12)   # thematic clusters

print("Config active:")
print(f"  Model:       {pipeline_core.CONFIG['LLM_MODEL']}  (temp={pipeline_core.CONFIG['LLM_TEMPERATURE']})")
print(f"  L1 range:    {pipeline_core.CONFIG['L1_CODES_RANGE']}")
print(f"  L2 range:    {pipeline_core.CONFIG['L2_CODES_RANGE']}")
print(f"  L3 range:    {pipeline_core.CONFIG['L3_CODES_RANGE']}")
print(f"  Clusters:    {pipeline_core.CONFIG['CLUSTERS_RANGE']}")

In [ ]:
# ── C1: Imports & env ────────────────────────────────────────────────────────
import os
import json
from collections import defaultdict
from dotenv import load_dotenv
import pipeline_core

load_dotenv(pipeline_core.KEYS_ENV)

loaded_keys = [k for k in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"] if os.getenv(k)]
print(f"Env loaded from '{pipeline_core.KEYS_ENV}'")
print(f"Keys found: {loaded_keys if loaded_keys else 'none — check keys.env'}")

In [ ]:
# ── C2: LLM client ───────────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C2" there to find/edit the implementation).
# To swap providers: edit pipeline_core.CONFIG["LLM_PROVIDER"] in C0 above.
# verify=False   — bypasses corporate TLS inspection (API key still authenticates).
# trust_env=False — ignores any HTTPS_PROXY / HTTP_PROXY env vars; connect directly.

from pipeline_core import call_llm

cfg = pipeline_core.CONFIG
print(f"LLM client ready: {cfg['LLM_PROVIDER']} / {cfg['LLM_MODEL']}  (temperature={cfg['LLM_TEMPERATURE']})")

In [ ]:
# ── C3: Data ingestion ───────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C3"). Edit there to adapt to future input formats.

from pipeline_core import load_interviews

interviews = load_interviews()

print(f"Loaded {len(interviews)} interviews\n")
for iv in interviews:
    print(f"  {iv['interview_id'][:8]}...  {len(iv['qa_pairs'])} Q&A pairs")

print("\nSample (first interview, first 2 pairs):")
for qa in interviews[0]["qa_pairs"][:2]:
    print(f"  [t{qa['turn_number']}] Q: {qa['question'][:65]}")
    print(f"         A: {qa['answer'][:65]}")
    print()

In [ ]:
# ── C4: Anonymizer ───────────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C4"). PII patterns are editable there.

from pipeline_core import anonymize

sample_raw  = interviews[0]["qa_pairs"][0]["answer"]
sample_anon = anonymize(sample_raw)
print("Anonymizer ready.")
print(f"  Before: {sample_raw[:80]}")
print(f"  After:  {sample_anon[:80]}")

In [ ]:
# ── C5: DB init ──────────────────────────────────────────────────────────────
# PK = interview_question_id (e.g. "6abc1e3b_t1").  Shape is fixed after this cell.

db = {}
for iv in interviews:
    prefix = iv["interview_id"][:8]
    for qa in iv["qa_pairs"]:
        iq_id = f"{prefix}_t{qa['turn_number']}"
        db[iq_id] = {
            "interview_id":      iv["interview_id"],
            "turn_number":       qa["turn_number"],
            "question":          qa["question"],
            "anonymised_answer": anonymize(qa["answer"]),
            "l1_codes":          []
        }

print(f"DB initialised: {len(db)} entries")
sample_key = next(iter(db))
print(f"\nSample entry  ({sample_key}):")
for k, v in db[sample_key].items():
    print(f"  {k}: {str(v)[:80]!r}")

In [ ]:
# ── C6: Coder (L1) ───────────────────────────────────────────────────────────
# Per Q&A: 1-10 open codes. Prompt defined in pipeline_core.py (search "C6").

from pipeline_core import code_one_answer

for iq_id, entry in db.items():
    db[iq_id]["l1_codes"] = code_one_answer(entry["question"], entry["anonymised_answer"])

counts = [len(e["l1_codes"]) for e in db.values()]
print(f"L1 coding done: {len(db)} entries coded")
print(f"Codes/entry:  min={min(counts)}  max={max(counts)}  avg={sum(counts)/len(counts):.1f}")
sample_key = next(iter(db))
print(f"\nSample ({sample_key}):")
print(f"  Q: {db[sample_key]['question'][:70]}")
print(f"  Codes: {db[sample_key]['l1_codes']}")

In [ ]:
# ── C7: User Consolidator (L2) ───────────────────────────────────────────────
# Per interview: L1 codes → L2_CODES_RANGE consolidated codes with merge lineage.
# Prompt defined in pipeline_core.py (search "C7").

from pipeline_core import consolidate_l2

interview_store = {}

by_interview = defaultdict(list)
for iq_id, entry in db.items():
    by_interview[entry["interview_id"]].append(entry)

for interview_id, entries in by_interview.items():
    all_l1 = [code for entry in entries for code in entry["l1_codes"]]
    interview_store[interview_id] = {"l2_codes": consolidate_l2(all_l1)}

print(f"L2 consolidation done: {len(interview_store)} interviews")
for iv_id, store in interview_store.items():
    print(f"  {iv_id[:8]}...  {len(store['l2_codes'])} L2 codes")

sample_iv = next(iter(interview_store))
print(f"\nSample L2 codes ({sample_iv[:8]}...):")
for item in interview_store[sample_iv]["l2_codes"][:3]:
    print(f"  {item['code']!r:42}  <- {item['merged_from_l1']}")

In [ ]:
# ── C8: Global Consolidator (L3) ─────────────────────────────────────────────
# All L2 codes → L3_CODES_RANGE consolidated codes with merge lineage.
# Prompt defined in pipeline_core.py (search "C8").

from pipeline_core import consolidate_l3

n_interviews = len(interview_store)
all_l2_codes = [item["code"] for store in interview_store.values() for item in store["l2_codes"]]

global_store = {"l3_codes": consolidate_l3(all_l2_codes, n_interviews)}

n_l3 = len(global_store["l3_codes"])
l3_range = pipeline_core.CONFIG["L3_CODES_RANGE"]
print(f"L3 consolidation done: {n_l3} codes  (range: {l3_range[0]}-{l3_range[1]})")
print("\nFirst 10 L3 codes:")
for item in global_store["l3_codes"][:10]:
    print(f"  {item['code']!r:42}  <- {item['merged_from_l2']}")

In [ ]:
# ── C9: Theme Clustering + Lineage ───────────────────────────────────────────
# L3 codes → named clusters. Prompt defined in pipeline_core.py (search "C9").
# Lineage (PK = cluster name) is built in this cell — it is orchestration, not a function.

from pipeline_core import cluster_l3_codes

l3_list    = [item["code"] for item in global_store["l3_codes"]]
cluster_map = cluster_l3_codes(l3_list)   # {cluster_name: [l3_code, ...]}

# ── Build lineage ─────────────────────────────────────────────────────────────
l3_to_l2 = {item["code"]: item["merged_from_l2"] for item in global_store["l3_codes"]}

l2_to_interview = {}
l2_to_l1        = {}
for interview_id, store in interview_store.items():
    for item in store["l2_codes"]:
        l2_to_interview[item["code"]] = interview_id
        l2_to_l1[item["code"]]        = item["merged_from_l1"]

l1_to_iq_ids = defaultdict(list)
for iq_id, entry in db.items():
    for code in entry["l1_codes"]:
        l1_to_iq_ids[code].append(iq_id)

lineage  = {}
clusters = {}

for name, l3_codes_in_cluster in cluster_map.items():
    seen_l2 = {}
    seen_l1 = {}
    for l3_code in l3_codes_in_cluster:
        for l2_code in l3_to_l2.get(l3_code, []):
            seen_l2[l2_code] = l2_to_interview.get(l2_code, "unknown")
            for l1_code in l2_to_l1.get(l2_code, []):
                if l1_code not in seen_l1 and l1_to_iq_ids[l1_code]:
                    seen_l1[l1_code] = l1_to_iq_ids[l1_code][0]

    lineage[name] = {
        "l3_codes": l3_codes_in_cluster,
        "l2_codes": [{"code": c, "interview_id": iv} for c, iv in seen_l2.items()],
        "l1_codes": [{"code": c, "interview_question_id": iq} for c, iq in seen_l1.items()],
    }
    clusters[name] = {"l3_codes": l3_codes_in_cluster, "story": ""}

cl_range = pipeline_core.CONFIG["CLUSTERS_RANGE"]
print(f"Clustering done: {len(clusters)} clusters (range: {cl_range[0]}-{cl_range[1]})\n")
for name, lin in lineage.items():
    iq_ids = {e["interview_question_id"] for e in lin["l1_codes"]}
    print(f"  {name}")
    shown  = sorted(iq_ids)[:5]
    suffix = "..." if len(iq_ids) > 5 else ""
    print(f"    Sources: {shown}{suffix}")

In [ ]:
# ── C10: LLM Explainer ───────────────────────────────────────────────────────
# Per cluster: structured finding (summary/quotes/category).
# Prompts defined in pipeline_core.py (search "C10").

from pipeline_core import explain_cluster, propose_experiments

for name, lin in lineage.items():
    iq_ids = list({e["interview_question_id"] for e in lin["l1_codes"]})

    qa_lines = []
    for iq_id in iq_ids:
        if iq_id in db:
            e = db[iq_id]
            qa_lines.append(f"Q: {e['question']}\nA: {e['anonymised_answer']}")

    qa_pairs_text = "\n\n".join(qa_lines) if qa_lines else "(no source answers found)"
    result        = explain_cluster(name, clusters[name]["l3_codes"], qa_pairs_text)
    voice_count   = len({e["interview_id"] for e in lin["l2_codes"]})

    clusters[name].update({
        "category":    result.get("category", "mixed"),
        "summary":     result.get("summary", ""),
        "quotes":      result.get("quotes", []),
        "tag":         result.get("tag", ""),
        "story":       result.get("summary", ""),
        "voice_count": voice_count,
    })
    print(f"[{result.get('category', '?')}]  {name}  ·  {result.get('tag', '')}  ·  {voice_count} voices")

# ── Experiment proposals ──────────────────────────────────────────────────────
needs_attention = [
    (name, data) for name, data in clusters.items()
    if data.get("category") in ("needs_work", "mixed")
]
experiments = propose_experiments(needs_attention)

print(f"\nExperiment proposals: {len(experiments)}")
for exp in experiments:
    print(f"  · {exp['title']}  ({exp.get('tag', '')})")

In [ ]:
# ── C11: Persist ─────────────────────────────────────────────────────────────
# Write all data structures to OUTPUT_DIR as pretty-printed JSON.
# Re-run after any cell to checkpoint progress.

import datetime

os.makedirs(pipeline_core.OUTPUT_DIR, exist_ok=True)

to_save = {
    "db":              db,
    "interview_store": interview_store,
    "global_store":    global_store,
    "lineage":         lineage,
    "clusters":        clusters,
    "experiments":     globals().get("experiments", []),
}

for name, data in to_save.items():
    path = os.path.join(pipeline_core.OUTPUT_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(path) / 1024
    print(f"  Saved: {path}  ({size_kb:.1f} KB)")

# ── run_meta.json — attributes results to this model run ──────────────────────
run_meta = {
    "model":        pipeline_core.CONFIG["LLM_MODEL"],
    "temperature":  pipeline_core.CONFIG["LLM_TEMPERATURE"],
    "timestamp":    datetime.datetime.utcnow().isoformat() + "Z",
    "n_interviews": len(interviews),
}
meta_path = os.path.join(pipeline_core.OUTPUT_DIR, "run_meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)
print(f"  Saved: {meta_path}")

print(f"\nAll files written to '{pipeline_core.OUTPUT_DIR}'")

In [ ]:
# ── C12: Report renderer ─────────────────────────────────────────────────────
# Renders the full HR report as styled HTML directly in the notebook output.

from IPython.display import display, HTML
import html as _html

def H(s): return _html.escape(str(s))

# Plain string — no f-string brace escaping needed for CSS
REPORT_CSS = """<style>
.sp{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,sans-serif;max-width:740px;margin:0 auto;color:#111}
.sp-hdr{display:flex;align-items:center;gap:14px;padding:22px 0 18px;border-bottom:2px solid #000;margin-bottom:28px}
.sp-logo{width:40px;height:40px;background:#000;border-radius:9px;color:#fff;font-weight:800;font-size:14px;letter-spacing:-1px;display:flex;align-items:center;justify-content:center;flex-shrink:0}
.sp h1{font-size:20px;font-weight:700;margin:0}
.sp-meta{font-size:12px;color:#888;margin:2px 0 0}
.sp h2{font-size:16px;font-weight:700;margin:32px 0 16px}
.sp h2:first-of-type{margin-top:0}
hr.sp-hr{border:none;border-top:1px solid #ebebeb;margin:18px 0}
.sp-card{margin-bottom:18px}
.sp-hl{font-size:16px;font-weight:600;margin:0 0 3px}
.sp-sub{font-size:12px;color:#777;margin:0 0 8px}
.sp-badge{background:#f2f2f2;border-radius:20px;padding:2px 9px;font-size:11px;font-weight:500}
details.sp-d{border:1px solid #e8e8e8;border-radius:7px}
details.sp-d>summary{padding:8px 13px;cursor:pointer;font-size:12px;font-weight:500;color:#666;list-style:none;user-select:none}
details.sp-d>summary::-webkit-details-marker{display:none}
details.sp-d>summary::before{content:'▶ ';font-size:9px}
details[open].sp-d>summary::before{content:'▼ '}
.sp-db{padding:2px 14px 14px}
.sp-db h4{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:.5px;color:#bbb;margin:12px 0 5px}
.sp-db p{font-size:13px;color:#333;line-height:1.7}
.sp-ql{list-style:none;padding:0;margin:0}
.sp-ql li{font-size:13px;color:#444;padding:6px 0 6px 11px;border-left:3px solid #e0e0e0;margin-bottom:6px}
.sp-exp-hl{font-size:15px;font-weight:600;margin:0 0 3px}
.sp-about{background:#f8f8f8;border-radius:9px;padding:18px 22px;margin-top:32px}
.sp-about h3{font-size:14px;font-weight:700;margin:0 0 8px}
.sp-about p,.sp-about ol{font-size:12px;color:#666;line-height:1.7}
.sp-about ol{padding-left:16px;margin-top:6px}
.sp-foot{text-align:center;font-size:11px;color:#ccc;padding:28px 0 8px}
</style>"""

def _card(name, data):
    hl = H(name)
    tg = H(data.get("tag", ""))
    v  = data.get("voice_count", 0)
    sm = H(data.get("summary", ""))
    ql = "".join(f'<li>{H(q)}</li>' for q in data.get("quotes", []))
    return (
        f'<div class="sp-card"><p class="sp-hl">{hl}</p>'
        f'<p class="sp-sub"><span class="sp-badge">{tg}</span>'
        f' &middot; {v} voice{"s" if v != 1 else ""}</p>'
        f'<details class="sp-d"><summary>Read summary and quotes</summary>'
        f'<div class="sp-db"><h4>Summary</h4><p>{sm}</p>'
        f'<h4>Paraphrased quotes</h4>'
        f'<ol class="sp-ql">{ql}</ol></div></details></div>'
        f'<hr class="sp-hr">'
    )

def _exp_card(exp):
    t = H(exp.get("title", ""))
    s = H(exp.get("summary", ""))
    r = H(exp.get("rationale", ""))
    g = H(exp.get("tag", ""))
    return (
        f'<div class="sp-card"><p class="sp-exp-hl">{t}</p>'
        f'<p class="sp-sub"><span class="sp-badge">{g}</span> {s}</p>'
        f'<details class="sp-d"><summary>Read rationale</summary>'
        f'<div class="sp-db"><p>{r}</p></div></details></div>'
    )

_exps = globals().get("experiments", [])

by_cat = {"working_well": [], "needs_work": [], "mixed": []}
for name, data in clusters.items():
    by_cat.get(data.get("category", "mixed"), by_cat["mixed"]).append((name, data))

body = ""
if by_cat["working_well"]:
    body += "<h2>What&#x2019;s working well</h2>"
    for n, d in by_cat["working_well"]:
        body += _card(n, d)
if by_cat["needs_work"]:
    body += "<h2>What needs work</h2>"
    for n, d in by_cat["needs_work"]:
        body += _card(n, d)
if by_cat["mixed"]:
    body += "<h2>Mixed signals &amp; tensions</h2>"
    for n, d in by_cat["mixed"]:
        body += _card(n, d)
if _exps:
    body += "<h2>Experiments</h2>"
    for exp in _exps:
        body += _exp_card(exp)

n_iv = len(interviews)
body += (
    '<div class="sp-about">'
    '<h3>Spradley guide: How the analysis works</h3>'
    f'<p>Insights are based on qualitative analysis of {n_iv} employee AI interview transcripts. '
    'Rather than survey scores, we surface <strong>patterns</strong> from what people say.</p>'
    '<ol>'
    '<li>Transcripts are clustered to identify recurring patterns across the dataset.</li>'
    '<li>Patterns are distilled into clear findings.</li>'
    '<li>Findings are labelled based on matching workplace themes.</li>'
    '<li>Paraphrased quotes give context while preserving anonymity.</li>'
    '</ol></div>'
)

out = (
    REPORT_CSS
    + '<div class="sp">'
    + '<div class="sp-hdr">'
    + '<div class="sp-logo">SP</div>'
    + '<div>'
    + '<h1>Employee Insights Report</h1>'
    + f'<p class="sp-meta">Spradley &middot; {len(clusters)} themes &middot; {n_iv} interviews</p>'
    + '</div></div>'
    + body
    + '<div class="sp-foot">Spradley &middot; app.spradley.io</div>'
    + '</div>'
)

display(HTML(out))

In [ ]:
# ── C13: Web application ─────────────────────────────────────────────────────
# Writes pipeline_output/report.html and opens it in the default browser.
# Tab 1 — Report: findings grouped by category, expandable summaries + quotes, experiments.
# Tab 2 — Lineage: fully collapsible tree  cluster → L3 → L2 → L1 → source Q&A.

import webbrowser, html as _html, os

def H(s): return _html.escape(str(s))

# ── CSS (plain string — brace escaping not needed here) ───────────────────────
APP_CSS = """
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif;
     background:#fff;color:#111;font-size:15px;line-height:1.6}

.app-header{display:flex;align-items:center;gap:14px;padding:14px 40px;
            border-bottom:1px solid #e5e5e5;position:sticky;top:0;
            background:#fff;z-index:100}
.logo{width:36px;height:36px;background:#000;border-radius:8px;flex-shrink:0;
      color:#fff;font-weight:800;font-size:13px;letter-spacing:-1px;
      display:flex;align-items:center;justify-content:center}
.logo-img{width:36px;height:36px;border-radius:8px;object-fit:contain;flex-shrink:0}
.app-title{font-size:16px;font-weight:700}
.app-meta{font-size:12px;color:#999;margin-top:1px}
.tabs{margin-left:auto;display:flex;gap:4px}
.tab-btn{padding:6px 15px;border:1px solid #ddd;background:#fff;border-radius:6px;
         font-size:13px;font-weight:500;cursor:pointer;color:#555;transition:background .1s}
.tab-btn:hover{background:#f5f5f5}
.tab-btn.active{background:#000;color:#fff;border-color:#000}

.tab-pane{display:none;max-width:760px;margin:0 auto;padding:36px 24px 60px}
.tab-pane.active{display:block}

h2{font-size:17px;font-weight:700;margin:36px 0 16px}
h2:first-of-type{margin-top:0}
.card{margin-bottom:18px}
.hl{font-size:16px;font-weight:600;margin-bottom:3px}
.meta{font-size:12px;color:#777;margin-bottom:8px}
.badge{background:#f2f2f2;border-radius:20px;padding:2px 9px;font-size:11px;font-weight:500}
hr.div{border:none;border-top:1px solid #ebebeb;margin:16px 0}

details.det>summary{padding:9px 13px;cursor:pointer;font-size:13px;font-weight:500;
                     color:#666;list-style:none;border:1px solid #e8e8e8;
                     border-radius:7px;user-select:none;display:block}
details[open].det>summary{border-bottom-left-radius:0;border-bottom-right-radius:0}
details.det>summary::-webkit-details-marker{display:none}
details.det>summary::before{content:'▶  ';font-size:9px}
details[open].det>summary::before{content:'▼  '}
.det-body{padding:4px 14px 14px;border:1px solid #e8e8e8;border-top:none;
          border-radius:0 0 7px 7px}
.det-body h4{font-size:10px;font-weight:700;text-transform:uppercase;
             letter-spacing:.5px;color:#bbb;margin:12px 0 5px}
.det-body p{font-size:14px;color:#333;line-height:1.7}
.quotes{list-style:none;padding:0;margin:0}
.quotes li{font-size:14px;color:#444;padding:6px 0 6px 12px;
           border-left:3px solid #e0e0e0;margin-bottom:7px}
.exp-hl{font-size:15px;font-weight:600;margin-bottom:3px}
.about{background:#f8f8f8;border-radius:10px;padding:20px 22px;margin-top:36px}
.about h3{font-size:14px;font-weight:700;margin-bottom:10px}
.about p,.about ol{font-size:13px;color:#666;line-height:1.65}
.about ol{padding-left:18px;margin-top:8px}
.footer{text-align:center;font-size:12px;color:#ccc;padding:32px 0 8px}

.lineage-intro{font-size:13px;color:#666;margin-bottom:24px;line-height:1.7;
               max-width:600px}
details.tr{margin-bottom:4px}
details.tr-cluster{border-left:3px solid #000;padding-left:16px;margin-bottom:14px}
details.tr-l3{border-left:2px solid #c8d8ff;padding-left:14px;margin:4px 0}
details.tr-l2{border-left:2px solid #c8f0d8;padding-left:14px;margin:4px 0}
details.tr-l1{border-left:2px solid #ffe8c8;padding-left:14px;margin:4px 0}
details.tr>summary{list-style:none;cursor:pointer;padding:5px 4px;
                    display:flex;align-items:center;gap:7px;flex-wrap:wrap;
                    user-select:none}
details.tr>summary::-webkit-details-marker{display:none}
details[open].tr>summary{color:#000}
details.tr-cluster>summary{font-size:15px;font-weight:600;padding:7px 4px}
.lv{border-radius:4px;font-size:10px;font-weight:700;padding:1px 6px;
    flex-shrink:0;letter-spacing:.3px}
.lv-3{background:#e8f0fe;color:#1a56db}
.lv-2{background:#dcfce7;color:#15803d}
.lv-1{background:#fef3c7;color:#b45309}
.lv-dot{width:8px;height:8px;border-radius:50%;flex-shrink:0}
.lv-dot-g{background:#16a34a}
.lv-dot-r{background:#dc2626}
.lv-dot-a{background:#d97706}
.int-tag{font-size:11px;color:#aaa;font-family:monospace}
.src-n{margin-left:auto;font-size:11px;color:#bbb;white-space:nowrap}
.tr-body{padding:6px 0 2px 8px}
.qa-blk{background:#fafafa;border-radius:6px;padding:10px 12px;margin-bottom:8px}
.iq-tag{font-size:10px;font-family:monospace;background:#f0f0f0;border-radius:4px;
        padding:1px 5px;display:inline-block;margin-bottom:5px;color:#999}
.qa-ln{font-size:13px;color:#444;margin-bottom:3px}
.empty{font-size:12px;color:#ccc;font-style:italic;padding:4px 0}
"""

# ── Logo: look in assets/, referenced with relative path ../assets/ from report ──
_logo_candidates = ["spradley_logo.png", "spradley_logo.svg", "logo.png", "logo.svg"]
_logo_file = next(
    (f for f in _logo_candidates if os.path.exists(os.path.join("assets", f))),
    None
)
if _logo_file:
    _logo_html = f'<img src="../assets/{H(_logo_file)}" class="logo-img" alt="Spradley">'
else:
    _logo_html = '<div class="logo">SP</div>'

# ── Report tab ────────────────────────────────────────────────────────────────

_exps = globals().get("experiments", [])

def _rcard(name, data):
    hl = H(name)
    tg = H(data.get("tag", ""))
    v  = data.get("voice_count", 0)
    sm = H(data.get("summary", ""))
    ql = "".join(f"<li>{H(q)}</li>" for q in data.get("quotes", []))
    return (
        f'<div class="card"><p class="hl">{hl}</p>'
        f'<p class="meta"><span class="badge">{tg}</span>'
        f' &middot; {v} voice{"s" if v != 1 else ""}</p>'
        f'<details class="det"><summary>Read summary and quotes</summary>'
        f'<div class="det-body"><h4>Summary</h4><p>{sm}</p>'
        f'<h4>Paraphrased quotes</h4>'
        f'<ol class="quotes">{ql}</ol></div></details></div>'
        f'<hr class="div">'
    )

def _ecard(exp):
    t = H(exp.get("title", ""))
    s = H(exp.get("summary", ""))
    r = H(exp.get("rationale", ""))
    g = H(exp.get("tag", ""))
    return (
        f'<div class="card"><p class="exp-hl">{t}</p>'
        f'<p class="meta"><span class="badge">{g}</span> {s}</p>'
        f'<details class="det"><summary>Read rationale</summary>'
        f'<div class="det-body"><p>{r}</p></div></details></div>'
    )

_by_cat = {"working_well": [], "needs_work": [], "mixed": []}
for _n, _d in clusters.items():
    _by_cat.get(_d.get("category", "mixed"), _by_cat["mixed"]).append((_n, _d))

_report = ""
if _by_cat["working_well"]:
    _report += "<h2>What&#x2019;s working well</h2>"
    for _n, _d in _by_cat["working_well"]:
        _report += _rcard(_n, _d)
if _by_cat["needs_work"]:
    _report += "<h2>What needs work</h2>"
    for _n, _d in _by_cat["needs_work"]:
        _report += _rcard(_n, _d)
if _by_cat["mixed"]:
    _report += "<h2>Mixed signals &#x26; tensions</h2>"
    for _n, _d in _by_cat["mixed"]:
        _report += _rcard(_n, _d)
if _exps:
    _report += "<h2>Experiments</h2>"
    for _exp in _exps:
        _report += _ecard(_exp)

_n_iv = len(interviews)
_report += (
    '<div class="about"><h3>Spradley guide: How the analysis works</h3>'
    f'<p>Insights are based on qualitative analysis of {_n_iv} employee AI interview '
    'transcripts. Rather than survey scores, we surface <strong>patterns</strong> '
    'from what people say.</p>'
    '<ol>'
    '<li>Transcripts are clustered to identify recurring patterns.</li>'
    '<li>Patterns are distilled into clear findings.</li>'
    '<li>Findings are labelled based on matching workplace themes.</li>'
    '<li>Paraphrased quotes give context while preserving anonymity.</li>'
    '</ol></div>'
    '<div class="footer">Spradley &middot; app.spradley.io</div>'
)

# ── Lineage tab ───────────────────────────────────────────────────────────────

_l3_map = {item["code"]: item for item in global_store["l3_codes"]}
_l2_map = {}
for _iv_id, _store in interview_store.items():
    for _item in _store["l2_codes"]:
        _l2_map[_item["code"]] = {"interview_id": _iv_id, **_item}

_dot_cls = {"working_well": "lv-dot-g", "needs_work": "lv-dot-r", "mixed": "lv-dot-a"}

def _ltree():
    out = ""
    for _cname, _lin in lineage.items():
        _n_src   = len({e["interview_question_id"] for e in _lin["l1_codes"]})
        _cl_data = clusters.get(_cname, {})
        _cl_dot  = _dot_cls.get(_cl_data.get("category", "mixed"), "lv-dot-a")
        _cl_tag  = H(_cl_data.get("tag", ""))

        _l3h = ""
        for _l3c in _lin["l3_codes"]:
            _l3i  = _l3_map.get(_l3c, {})
            _ml2  = _l3i.get("merged_from_l2", [])
            _l2h  = ""
            for _l2c in _ml2:
                _l2i  = _l2_map.get(_l2c, {})
                _ivid = _l2i.get("interview_id", "unknown")
                _ml1  = _l2i.get("merged_from_l1", [])
                _l1h  = ""
                for _l1c in _ml1:
                    _iqs   = [iq for iq, e in db.items() if _l1c in e.get("l1_codes", [])]
                    _qah   = ""
                    for _iq in _iqs:
                        _e = db[_iq]
                        _qah += (
                            f'<div class="qa-blk">'
                            f'<span class="iq-tag">{H(_iq)}</span>'
                            f'<p class="qa-ln"><strong>Q:</strong> {H(_e["question"])}</p>'
                            f'<p class="qa-ln"><strong>A:</strong> {H(_e["anonymised_answer"])}</p>'
                            f'</div>'
                        )
                    _l1h += (
                        f'<details class="tr tr-l1">'
                        f'<summary><span class="lv lv-1">L1</span> {H(_l1c)}</summary>'
                        f'<div class="tr-body">'
                        f'{_qah or "<p class=empty>No source Q&amp;A found.</p>"}'
                        f'</div></details>'
                    )
                _l2h += (
                    f'<details class="tr tr-l2">'
                    f'<summary><span class="lv lv-2">L2</span> {H(_l2c)}'
                    f' <span class="int-tag">{H(_ivid[:8])}</span></summary>'
                    f'<div class="tr-body">'
                    f'{_l1h or "<p class=empty>No L1 codes.</p>"}'
                    f'</div></details>'
                )
            _l3h += (
                f'<details class="tr tr-l3">'
                f'<summary><span class="lv lv-3">L3</span> {H(_l3c)}</summary>'
                f'<div class="tr-body">'
                f'{_l2h or "<p class=empty>No L2 codes.</p>"}'
                f'</div></details>'
            )
        out += (
            f'<details class="tr tr-cluster">'
            f'<summary>{H(_cname)}'
            f' <span class="lv-dot {_cl_dot}"></span>'
            f' <span class="badge">{_cl_tag}</span>'
            f' <span class="src-n">{_n_src} sources</span></summary>'
            f'<div class="tr-body">'
            f'{_l3h or "<p class=empty>No L3 codes.</p>"}'
            f'</div></details>'
        )
    return out

_lineage_tab = (
    '<p class="lineage-intro">Expand any cluster to trace a finding back to its source '
    'interview answers. Each level is independently collapsible. Open only '
    'what you need without losing the rest.</p>'
    + _ltree()
)

# ── Assemble full HTML ────────────────────────────────────────────────────────

_meta = f"Spradley · {len(clusters)} themes · {_n_iv} interviews"

_full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Spradley: Employee Insights</title>
<style>{APP_CSS}</style>
</head>
<body>

<header class="app-header">
  {_logo_html}
  <div>
    <div class="app-title">Employee Insights Report</div>
    <div class="app-meta">{H(_meta)}</div>
  </div>
  <nav class="tabs">
    <button class="tab-btn active" onclick="showTab('report',this)">Report</button>
    <button class="tab-btn" onclick="showTab('lineage',this)">Data Lineage</button>
  </nav>
</header>

<div id="pane-report" class="tab-pane active">
{_report}
</div>

<div id="pane-lineage" class="tab-pane">
{_lineage_tab}
</div>

<script>
function showTab(name,btn){{
  document.querySelectorAll('.tab-pane').forEach(function(el){{el.classList.remove('active');}});
  document.querySelectorAll('.tab-btn').forEach(function(el){{el.classList.remove('active');}});
  document.getElementById('pane-'+name).classList.add('active');
  btn.classList.add('active');
}}
</script>

</body>
</html>"""

# ── Write and open ────────────────────────────────────────────────────────────
_out_path = os.path.join(OUTPUT_DIR, "report.html")
with open(_out_path, "w", encoding="utf-8") as _f:
    _f.write(_full_html)

_abs = os.path.abspath(_out_path)
print(f"Written: {_abs}")
webbrowser.open("file:///" + _abs.replace(os.sep, "/"))
print("Opened in browser.")